# 01 — Geometric Brownian Motion (GBM)

**What this notebook is:** a complete start-to-finish guide for the GBM stock-price model, plus an interactive Monte Carlo playground.

**Theory handbook (all models):** [`00_MODELS_EXPLAINED.ipynb`](00_MODELS_EXPLAINED.ipynb)

**Your project data (already prepared):**
- Prices: `../data/equity/prices_clean.csv`
- Log returns: `../data/equity/log_returns_all.csv`, `../data/equity/log_returns_by_regime.csv`
- Summary stats: `../data/equity/summary_stats.csv`

The playground at the bottom uses **synthetic** paths (sliders). The workflow above it explains how you would use **real** historical data to choose the parameters those sliders represent.


## 1. Model idea (keep this mental picture)

GBM says: price drifts smoothly with a constant average trend, and is shaken by constant-volatility noise. No jumps, no changing vol.

**Continuous (differential):**

$$dS_t = \mu S_t\, dt + \sigma S_t\, dW_t$$

**Simulation (not differential — what the code uses):**

$$S_{t+\Delta t} = S_t \exp\Big(\big(\mu - \tfrac{1}{2}\sigma^2\big)\Delta t + \sigma\sqrt{\Delta t}\, Z\Big),\quad Z\sim N(0,1)$$

| Symbol | Meaning |
|--------|---------|
| $S_t$ | stock price at time $t$ |
| $\mu$ | annual drift (average trend of log returns) |
| $\sigma$ | annual volatility (constant) |
| $\Delta t$ | time step in years (e.g. $1/252$ for one trading day) |
| $Z$ | fresh standard normal draw each step |

**Itô correction** $-\tfrac12\sigma^2$: needed so the exponential update stays consistent with the SDE and prices stay positive.

**What GBM misses:** fat tails, crash jumps, volatility clustering.


## 2. End-to-end workflow (first time → Monte Carlo paths)

Do these steps in order. Steps 1–4 use historical data once (or once per regime). Step 5 repeats inside the simulation.

| Step | What you do | Output |
|------|-------------|--------|
| **1. Collect prices** | Get adjusted close prices for your ticker (e.g. SPY) over a window / regime | Price series $S_0,S_1,\ldots$ |
| **2. Compute log returns** | $r_t = \ln(S_t / S_{t-1})$ | Return series |
| **3. Estimate parameters** | Compute $\hat\mu$ and $\hat\sigma$ from returns (formulas below) | Two numbers, used for the whole simulation |
| **4. Set simulation design** | Choose $S_0$ (today’s price), horizon $T$, steps $n$, number of paths | Grid: $\Delta t = T/n$ |
| **5. Simulate paths** | For each path, for each step: draw $Z$, update $S$ with the exp formula | Cloud of future prices |
| **6. Use the paths** | Histograms, VaR, option payoffs $\max(S_T-K,0)$, American exercise later | Research outputs |

**Regimes in this project** (fit separately if you want regime-specific $\mu,\sigma$):

| Regime | Window |
|--------|--------|
| Crisis | 2007–2009 |
| Normal | 2013–2014 |
| High vol | 2017–2018 |


## 3. How to calculate parameters from historical data

Assume daily returns and $N_{\text{days}}=252$ trading days per year.

### Step A — log returns

$$r_t = \ln\!\left(\frac{S_t}{S_{t-1}}\right)$$

In your files this is already in `log_returns_*.csv`.

### Step B — annual drift $\mu$ (estimated once)

$$\hat\mu = \bar{r}\times 252,\qquad \bar{r} = \frac{1}{n}\sum_{t=1}^{n} r_t$$

### Step C — annual volatility $\sigma$ (estimated once)

$$\hat\sigma = s_r\times\sqrt{252},\qquad s_r = \sqrt{\frac{1}{n-1}\sum_{t=1}^{n}(r_t-\bar{r})^2}$$

### Step D — starting price $S_0$

Usually the **last observed** adjusted close in your sample (or the price on the option’s trading date).

### Quick Python sketch (not run here — for later calibration)

```python
import pandas as pd
import numpy as np

r = pd.read_csv("../data/equity/log_returns_by_regime.csv")
sp = r[(r["ticker"]=="SPY") & (r["regime"]=="normal")]["log_return"]
mu_hat = sp.mean() * 252
sigma_hat = sp.std(ddof=1) * np.sqrt(252)
```

You can also read approximate moments from `../data/equity/summary_stats.csv` if columns match your ticker/regime.


## 4. Which parameters are constant vs which change along a path?

This is the key distinction for Monte Carlo.

### Calibrated once (constant for the whole simulation)

These are estimated from history **before** you simulate. They do **not** update step-by-step inside a path.

| Parameter | Role | Updates during a path? |
|-----------|------|------------------------|
| $\mu$ | drift | **No** — fixed |
| $\sigma$ | volatility | **No** — fixed (this is GBM’s defining assumption) |
| $S_0$ | start price | **No** — fixed input |
| $T$, $\Delta t$, $n_{\text{paths}}$ | design choices | **No** |

### Evolve along each Monte Carlo path

| Quantity | Role | Updates during a path? |
|----------|------|------------------------|
| $S_t$ | price | **Yes** — this *is* the path |
| $Z_t$ | random shock | **Yes** — new draw every step |

So in GBM the only “state” that forms the path is $S_t$. Volatility never changes.

### One simulation step (repeat for $t = 0, \Delta t, 2\Delta t, \ldots$)

1. Keep $\mu,\sigma$ as the same numbers you estimated.
2. Draw $Z\sim N(0,1)$.
3. Set $S_{t+\Delta t} = S_t\exp\big((\mu-\tfrac12\sigma^2)\Delta t + \sigma\sqrt{\Delta t}\,Z\big)$.


## 5. Interactive playground

Use the sliders to feel how $\mu$ and $\sigma$ change the cloud of paths and the return histogram.

When you later calibrate to SPY (or another ticker), plug your $\hat\mu$ and $\hat\sigma$ into these same sliders.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider

%matplotlib inline

def simulate_gbm(mu, sigma, S0, T, n_steps, n_paths, seed=42):
    rng = np.random.default_rng(seed)
    dt = T / n_steps
    z = rng.standard_normal((n_paths, n_steps))
    increments = (mu - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * z
    log_paths = np.cumsum(increments, axis=1)
    paths = S0 * np.exp(np.hstack([np.zeros((n_paths, 1)), log_paths]))
    t = np.linspace(0, T, n_steps + 1)
    return t, paths

def plot_gbm(mu=0.08, sigma=0.20, S0=100.0, T=1.0, n_steps=252, n_paths=50):
    t, paths = simulate_gbm(mu, sigma, S0, T, n_steps, n_paths)
    log_rets = np.diff(np.log(paths), axis=1).ravel()

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].plot(t, paths.T, alpha=0.35, lw=0.9)
    axes[0].plot(t, paths.mean(axis=0), color="black", lw=2, label="mean path")
    axes[0].set_title("GBM Monte Carlo paths")
    axes[0].set_xlabel("years")
    axes[0].set_ylabel("price")
    axes[0].legend(loc="upper left")

    axes[1].hist(log_rets, bins=60, density=True, alpha=0.75, color="steelblue")
    axes[1].set_title("Daily log-return distribution")
    axes[1].set_xlabel("log return")
    axes[1].set_ylabel("density")

    fig.suptitle(
        f"μ={mu:.2f}, σ={sigma:.2f}, S0={S0:.0f}, T={T:.2f}y, "
        f"paths={n_paths}, steps={n_steps}",
        fontsize=10,
    )
    plt.tight_layout()
    plt.show()

interact(
    plot_gbm,
    mu=FloatSlider(value=0.08, min=-0.20, max=0.40, step=0.01, description="μ (drift)"),
    sigma=FloatSlider(value=0.20, min=0.01, max=0.80, step=0.01, description="σ (vol)"),
    S0=FloatSlider(value=100.0, min=10.0, max=500.0, step=5.0, description="S0"),
    T=FloatSlider(value=1.0, min=0.1, max=5.0, step=0.1, description="T (years)"),
    n_steps=IntSlider(value=252, min=50, max=1000, step=10, description="steps"),
    n_paths=IntSlider(value=50, min=5, max=200, step=5, description="paths"),
);